# Case Study I – Telecom Billing Prediction

A telecom company has customers with different subscriptions (CRM options). Each subscription should produce a specific billing configuration (BIL items). Because of technical and human errors, customers with the same subscription are sometimes billed in different ways.

**Goal:** for each test customer, predict the full billing configuration (731 BIL items) from the subscription (745 CRM options).

**Main metric:** Exact Match Ratio (EMR) = share of customers with **all 731** BIL items correct. We also report the Hamming loss (share of single wrong items).

**Pipeline of this notebook:**

· 1–3. Load and validate the data
· 4. Exploratory analysis
· 5. Data treatment: two cleaning versions (A and B)
· 6. Dimensionality reduction analysis (MCA and SVD)
· 7. Validation design
· 8. Models: the same six models trained on each cleaning version
· 9. Model and cleaning selection
· 10. Final model: train on all data and predict the test set
· 11. Final check against `solution.csv`

Run the notebook from top to bottom. Files needed in the same folder: `train.csv`, `test.csv`, `solution.csv`.

### 1. Importing libraries

In [4]:
%pip install prince

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


`prince` is used for Multiple Correspondence Analysis (install it once with `%pip install prince`).

In [5]:
import re
import time

import numpy as np
import pandas as pd
import prince
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

### 2. Loading the training and test sets

All CRM and BIL columns contain only 0 and 1, so they are loaded directly as `uint8`. This uses 8 times less memory than the default integer type and makes every later step faster.

In [6]:
id_col = "MSISDN"


def binary_column_types(file_name):
    # Read only the header, then load every 0/1 column as uint8 and the ID as text
    columns = pd.read_csv(file_name, nrows=0).columns
    return {c: (str if c == id_col else np.uint8) for c in columns}


train = pd.read_csv("train.csv", dtype=binary_column_types("train.csv"))
test = pd.read_csv("test.csv", dtype=binary_column_types("test.csv"))

print(f"Train: {train.shape[0]:,} rows, {train.shape[1]:,} columns")
print(f"Test:  {test.shape[0]:,} rows, {test.shape[1]:,} columns")
train.head()

Train: 187,442 rows, 1,477 columns
Test:  97,100 rows, 746 columns


,MSISDN,CRM_BUSINESS_LINE_ORIG_Mobile,CRM_1519_PACK,CRM_1521_PACK,CRM_1522_PACK,CRM_1523_PACK,CRM_1524_PACK,CRM_1525_PACK,CRM_1526_PACK,CRM_1527_PACK,CRM_1528_PACK,CRM_1529_PACK,CRM_1530_PACK,CRM_1532_PACK,CRM_1533_PACK,CRM_1534_PACK,CRM_1536_PACK,CRM_1537_PACK,CRM_1538_PACK,CRM_1539_PACK,CRM_1540_PACK,CRM_1541_PACK,CRM_1542_PACK,CRM_1543_PACK,CRM_1545_PACK,CRM_1546_PACK,CRM_1547_PACK,CRM_1548_PACK,CRM_1549_PACK,CRM_1550_PACK,CRM_1551_PACK,CRM_1552_PACK,CRM_1553_PACK,CRM_1554_PACK,CRM_1555_PACK,CRM_1556_PACK,CRM_1557_PACK,CRM_1558_PACK,CRM_1559_PACK,CRM_1560_PACK,CRM_1562_PACK,CRM_1563_PACK,CRM_1564_PACK,CRM_1565_PACK,CRM_1566_PACK,CRM_1567_PACK,CRM_1568_PACK,CRM_1569_PACK,CRM_1570_PACK,CRM_1571_PACK,CRM_1573_PACK,CRM_1575_PACK,CRM_1576_PACK,CRM_1577_PACK,CRM_1578_PACK,CRM_1579_PACK,CRM_1581_PACK,CRM_1582_PACK,CRM_1583_PACK,CRM_1584_PACK,CRM_1585_PACK,CRM_1586_PACK,CRM_1587_PACK,CRM_1588_PACK,CRM_1589_PACK,CRM_1590_PACK,CRM_1591_PACK,CRM_1592_PACK,CRM_1593_PACK,CRM_1594_PACK,CRM_1595_PACK,CRM_1596_PACK,CRM_1598_PACK,CRM_1599_PACK,CRM_1601_PACK,CRM_1602_PACK,CRM_1603_PACK,CRM_1604_PACK,CRM_1605_PACK,CRM_1606_PACK,CRM_1608_PACK,CRM_1609_PACK,CRM_1612_PACK,CRM_1613_PACK,CRM_1614_PACK,CRM_1615_PACK,CRM_1616_PACK,CRM_1617_PACK,CRM_1619_PACK,CRM_1623_PACK,CRM_1625_PACK,CRM_1626_PACK,CRM_1627_PACK,CRM_1629_PACK,CRM_1630_PACK,CRM_1631_PACK,CRM_1635_PACK,CRM_1638_PACK,CRM_1639_PACK,CRM_1640_PACK,CRM_1641_PACK,CRM_1642_PACK,CRM_1643_PACK,CRM_1644_PACK,CRM_1645_PACK,CRM_1646_PACK,CRM_1647_PACK,CRM_1648_PACK,CRM_1653_PACK,CRM_1654_PACK,CRM_1658_PACK,CRM_1661_PACK,CRM_1662_PACK,CRM_1664_PACK,CRM_1665_PACK,CRM_1666_PACK,CRM_1668_PACK,CRM_1669_PACK,CRM_1670_PACK,CRM_1671_PACK,CRM_1675_PACK,CRM_1676_PACK,CRM_1678_PACK,CRM_1680_PACK,CRM_1683_PACK,CRM_1689_PACK,CRM_1690_PACK,CRM_1692_PACK,CRM_1702_PACK,CRM_1704_PACK,CRM_1705_PACK,CRM_1708_PACK,CRM_1709_PACK,CRM_1710_PACK,CRM_1715_PACK,CRM_2076_PACK,CRM_2077_PACK,CRM_2078_PACK,CRM_2079_PACK,CRM_2080_PACK,CRM_2081_PACK,CRM_2082_PACK,CRM_2083_PACK,CRM_2084_PACK,CRM_2085_PACK,CRM_2086_PACK,CRM_2087_PACK,CRM_2088_PACK,CRM_2089_PACK,CRM_2090_PACK,CRM_2091_PACK,CRM_2092_PACK,CRM_2093_PACK,CRM_2094_PACK,CRM_2095_PACK,CRM_2096_PACK,CRM_2097_PACK,CRM_2099_PACK,CRM_2100_PACK,CRM_2101_PACK,CRM_2102_PACK,CRM_2103_PACK,CRM_2104_PACK,CRM_2105_PACK,CRM_2106_PACK,CRM_2107_PACK,CRM_2109_PACK,CRM_2111_PACK,CRM_2112_PACK,CRM_2113_PACK,CRM_2114_PACK,CRM_2115_PACK,CRM_2116_PACK,CRM_2117_PACK,CRM_2118_PACK,CRM_2121_PACK,CRM_2123_PACK,CRM_2126_PACK,CRM_2127_PACK,CRM_2128_PACK,CRM_2129_PACK,CRM_2130_PACK,CRM_2132_PACK,CRM_2134_PACK,CRM_2135_PACK,CRM_2137_PACK,CRM_2138_PACK,CRM_2139_PACK,CRM_2140_PACK,CRM_SUBSCRIBER_TYPE_ORIG_0,CRM_SUBSCRIBER_TYPE_ORIG_Postpaid,CRM_SUBSCRIBER_TYPE_ORIG_Prepaid,CRM_SUBSCRIBER_STATUS_ORIG_0,CRM_SUBSCRIBER_STATUS_ORIG_Active,CRM_SUBSCRIBER_STATUS_ORIG_Barring,CRM_SUBSCRIBER_STATUS_ORIG_Block_1_Way,CRM_SUBSCRIBER_STATUS_ORIG_Block_2_Way,CRM_SUBSCRIBER_STATUS_ORIG_Deactive,CRM_SUBSCRIBER_STATUS_ORIG_Idle,CRM_SUBSCRIBER_STATUS_ORIG_Pre_Active,CRM_SUBSCRIBER_STATUS_ORIG_Pre_Deactivated,CRM_SUBSCRIBER_STATUS_ORIG_Pre_Expired,CRM_SUBSCRIBER_STATUS_ORIG_Suspend,CRM_2141_PACK,CRM_2142_PACK,CRM_2143_PACK,CRM_2146_PACK,CRM_2147_PACK,CRM_2148_PACK,CRM_2149_PACK,CRM_2153_PACK,CRM_2154_PACK,CRM_2155_PACK,CRM_2161_PACK,CRM_1531_PACK,CRM_1561_PACK,CRM_1597_PACK,CRM_1600_PACK,CRM_1607_PACK,CRM_1610_PACK,CRM_1611_PACK,CRM_1618_PACK,CRM_1620_PACK,CRM_1621_PACK,CRM_1622_PACK,CRM_1624_PACK,CRM_1633_PACK,CRM_1636_PACK,CRM_1637_PACK,CRM_1649_PACK,CRM_1650_PACK,CRM_1651_PACK,CRM_1652_PACK,CRM_1655_PACK,CRM_1656_PACK,CRM_1657_PACK,CRM_1660_PACK,CRM_1663_PACK,CRM_1667_PACK,CRM_1672_PACK,CRM_1673_PACK,CRM_1674_PACK,CRM_1677_PACK,CRM_1679_PACK,CRM_1682_PACK,CRM_1684_PACK,CRM_1685_PACK,CRM_1686_PACK,CRM_1687_PACK,CRM_1688_PACK,CRM_1691_PACK,CRM_1693_PACK,CRM_1694_PACK,CRM_1695_PACK,CRM_1696_PACK,CRM_1697_PACK,CRM_1698_PACK,CRM_1699_PACK,CRM_1700_PACK,CRM_1701_PACK,CRM_1706_PACK,CRM_1712_PACK,CRM_1714_PACK,CRM_1717_PA

### 3. Separate identifiers, CRM features, and Billing targets

This creates three logical groups:

· MSISDN: customer identifier
· crm_cols: 745 input features
· bil_cols: 731 output targets

The test set must have exactly the same CRM columns as the training set, unique customer IDs and only 0/1 values, because the model must receive the same feature structure during training and prediction.

In [7]:
crm_cols = [c for c in train.columns if c.startswith("CRM")]
bil_cols = [c for c in train.columns if c.startswith("BIL")]

assert len(crm_cols) == 745
assert len(bil_cols) == 731
assert list(train.columns) == [id_col] + crm_cols + bil_cols

# Test validation
assert list(test.columns) == [id_col] + crm_cols
assert test[id_col].notna().all() and not test[id_col].duplicated().any()
# uint8 values are never negative, so "maximum <= 1" means every value is 0 or 1
assert test[crm_cols].to_numpy().max() <= 1
assert train[crm_cols + bil_cols].to_numpy().max() <= 1

print("CRM features:", len(crm_cols), "| Billing targets:", len(bil_cols))
print("Train and test CRM columns match: True")

CRM features: 745 | Billing targets: 731
Train and test CRM columns match: True


### 4. Exploratory analysis

Each subscription (a full row of 745 CRM values) and each bill (a full row of 731 BIL values) gets one ID number, calculated with a hash. Rows with the same values get the same ID. This lets us count repeated and conflicting subscriptions quickly, without comparing 745 columns row by row.

A **broken profile** is a customer who does not have exactly one subscriber type and exactly one subscriber status.

In [8]:
PROFILE_GROUPS = ["CRM_SUBSCRIBER_TYPE", "CRM_SUBSCRIBER_STATUS",
                  "BIL_SUBSCRIBER_TYPE", "BIL_SUBSCRIBER_STATUS"]


def row_ids(data, columns):
    # One ID number per row: same 0/1 values -> same ID
    return pd.util.hash_pandas_object(data[columns], index=False).to_numpy()


def broken_profile_mask(data, groups):
    # True for rows that do not have exactly one "1" in each group of columns
    correct = np.ones(len(data), dtype=bool)
    for group in groups:
        group_cols = [c for c in data.columns if c.startswith(group)]
        correct &= data[group_cols].sum(axis=1).to_numpy() == 1
    return ~correct


train_crm_id = row_ids(train, crm_cols)
train_bil_id = row_ids(train, bil_cols)
test_crm_id = row_ids(test, crm_cols)

pairs = pd.DataFrame({"crm_id": train_crm_id, "bil_id": train_bil_id})
bills_per_subscription = pairs.groupby("crm_id")["bil_id"].nunique()
test_profile_groups = [g for g in PROFILE_GROUPS if g.startswith("CRM")]

overview = pd.Series({
    "Missing values (train)": int(train.isna().sum().sum()),
    "Missing values (test)": int(test.isna().sum().sum()),
    "Constant CRM columns in train": sum(train[c].min() == train[c].max() for c in crm_cols),
    "Constant CRM columns in test": sum(test[c].min() == test[c].max() for c in crm_cols),
    "Constant BIL columns in train": sum(train[c].min() == train[c].max() for c in bil_cols),
    "Fully duplicated rows (train)": int(pairs.duplicated().sum()),
    "Unique subscriptions (train)": int(pairs["crm_id"].nunique()),
    "Subscriptions with more than one bill": int((bills_per_subscription > 1).sum()),
    "Broken profiles (train)": int(broken_profile_mask(train, PROFILE_GROUPS).sum()),
    "Broken profiles (test, CRM side)": int(broken_profile_mask(test, test_profile_groups).sum()),
    "Unique subscriptions (test)": len(set(test_crm_id)),
    "Test subscriptions also seen in train": len(set(test_crm_id) & set(train_crm_id)),
}).to_frame("value")

constant_train_crm = [c for c in crm_cols if train[c].min() == train[c].max()]
print("Constant CRM columns in train:", constant_train_crm)
overview

Constant CRM columns in train: ['CRM_2396_PACK', 'CRM_2398_PACK']


,value
Missing values (train),0
Missing values (test),0
Constant CRM columns in train,2
Constant CRM columns in test,33
Constant BIL columns in train,0
Fully duplicated rows (train),119264
Unique subscriptions (train),67434
Subscriptions with more than one bill,617
Broken profiles (train),4728
"Broken profiles (test, CRM side)",0


**How to read it:**

· Many rows are exact copies, so the useful information is one row per subscription.
· Some subscriptions appear with more than one bill: these are the billing errors described in the problem.
· Almost no test subscription appears exactly in the training set. The models must learn general rules from the individual CRM options, not memorize full subscriptions.

#### 4.1 Strongest CRM–Billing associations

For two 0/1 variables, the Pearson correlation is called the **phi coefficient**. A value close to 1 means that the CRM option and the BIL item almost always appear together. This function is also used later by the association model (section 8).

In [9]:
def phi_matrix(features, targets, weights, chunk_size=8192):
    # Weighted phi coefficient between every CRM column and every BIL column.
    # The product is done in chunks of rows to limit memory use.
    weights = np.asarray(weights, dtype=np.float64)
    total = weights.sum()
    feature_sums = weights @ features
    target_sums = weights @ targets
    cross = np.zeros((features.shape[1], targets.shape[1]), dtype=np.float64)
    for start in range(0, len(features), chunk_size):
        stop = start + chunk_size
        x_chunk = features[start:stop].astype(np.float64) * weights[start:stop, None]
        cross += x_chunk.T @ targets[start:stop].astype(np.float64)
    numerator = total * cross - feature_sums[:, None] * target_sums[None, :]
    feature_var = feature_sums * (total - feature_sums)
    target_var = target_sums * (total - target_sums)
    denominator = np.sqrt(feature_var[:, None] * target_var[None, :])
    return np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)


train_phi = phi_matrix(train[crm_cols].to_numpy(), train[bil_cols].to_numpy(), np.ones(len(train)))
association_table = (
    pd.DataFrame(np.abs(train_phi), index=crm_cols, columns=bil_cols)
    .stack()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
association_table.columns = ["CRM_feature", "Billing_target", "absolute_phi"]
association_table

,CRM_feature,Billing_target,absolute_phi
0,CRM_1533_PACK,BIL_3934_PACK,0.995789
1,CRM_1533_PACK,BIL_3933_PACK,0.995703
2,CRM_1542_PACK,BIL_3933_PACK,0.995575
3,CRM_1542_PACK,BIL_3934_PACK,0.995532
4,CRM_1541_PACK,BIL_3927_PACK,0.995523
5,CRM_1529_PACK,BIL_3921_PACK,0.995308
6,CRM_1532_PACK,BIL_3922_PACK,0.995248
7,CRM_1538_PACK,BIL_3935_PACK,0.995182
8,CRM_2088_PACK,BIL_3744_PACK,0.995107
9,CRM_2087_PACK,BIL_3745_PACK,0.995042


### 5. Data treatment: two cleaning versions

Both versions solve the main problem the same way: **one row per subscription**, keeping the bill that most customers with that subscription have (majority vote). They differ in three decisions:

| Decision | Cleaning A – strict | Cleaning B – majority |
|---|---|---|
| CRM columns that are constant in train | removed (they carry no information for training) | kept (they vary in the test set) |
| Broken profiles | removed | kept |
| Tie in the vote (e.g. 1 customer vs 1 customer) | subscription removed | the most frequent bill seen first is kept |

Each kept row also stores its **weight**: the number of customers that support it. Only the association models (section 8) use the weights.

The cleaning is written as functions, so exactly the same treatment can be applied to the training part of the validation split (section 7) and to the full training set (section 10). The test set is never cleaned by rows: every test customer needs a prediction.

In [10]:
def majority_vote(rows, feature_cols, target_cols, drop_ties):
    # Keep one row per subscription: the bill that most customers with it have
    pairs = pd.DataFrame({
        "crm_id": row_ids(rows, feature_cols),
        "bil_id": row_ids(rows, target_cols),
        "row": np.arange(len(rows)),
    })
    # Count every (subscription, bill) pair, in order of first appearance
    counts = (
        pairs.groupby(["crm_id", "bil_id"], sort=False)
        .agg(weight=("row", "size"), first_row=("row", "first"))
        .reset_index()
    )
    # Most frequent bill first; the stable sort keeps the first-seen bill on ties
    counts = counts.sort_values("weight", ascending=False, kind="stable")
    if drop_ties:
        top = counts.groupby("crm_id")["weight"].transform("max")
        winners_per_subscription = (counts["weight"] == top).groupby(counts["crm_id"]).transform("sum")
        counts = counts[winners_per_subscription == 1]
    winners = counts.drop_duplicates("crm_id", keep="first")
    cleaned = rows.iloc[winners["first_row"].to_numpy()][feature_cols + target_cols].reset_index(drop=True)
    return cleaned, winners["weight"].to_numpy(dtype=np.float64)


def clean_version_a(rows):
    # Strict: drop constant CRM columns, broken profiles and tied subscriptions
    features = [c for c in crm_cols if rows[c].min() != rows[c].max()]
    rows = rows[~broken_profile_mask(rows, PROFILE_GROUPS)]
    cleaned, weights = majority_vote(rows, features, bil_cols, drop_ties=True)
    return cleaned, weights, features


def clean_version_b(rows):
    # Majority: keep every CRM column and every row; ties keep the first-seen bill
    features = list(crm_cols)
    cleaned, weights = majority_vote(rows, features, bil_cols, drop_ties=False)
    return cleaned, weights, features


CLEANINGS = {
    "A – strict": clean_version_a,
    "B – majority": clean_version_b,
}

cleaning_summary = []
for cleaning_name, cleaning in CLEANINGS.items():
    cleaned, weights, features = cleaning(train)
    assert not pd.Series(row_ids(cleaned, features)).duplicated().any()
    cleaning_summary.append({
        "Cleaning": cleaning_name,
        "Rows before": len(train),
        "Rows after": len(cleaned),
        "CRM features": len(features),
        "Billing targets": len(bil_cols),
        "Customers represented (sum of weights)": int(weights.sum()),
    })
pd.DataFrame(cleaning_summary)

,Cleaning,Rows before,Rows after,CRM features,Billing targets,Customers represented (sum of weights)
0,A – strict,187442,62622,743,731,181295
1,B – majority,187442,67434,745,731,186127


### 6. Dimensionality reduction analysis: MCA and SVD

Could we summarize the 745 CRM options into fewer columns before modeling? We test two techniques:

· **MCA (Multiple Correspondence Analysis):** designed for categorical variables such as 0/1 options.
· **TruncatedSVD:** works directly on the sparse 0/1 matrix.

For each variable we calculate the **squared cosine (cos²)**: the share of the variable that is represented by the kept components (0 = not represented, 1 = fully represented). A variable is **well represented** if cos² ≥ 0.5. Both techniques keep the same number of components (100), so they are compared on the same basis.

The analysis uses one row per unique subscription of the training set and the CRM columns that are not constant.

In [11]:
N_COMPONENTS = 100
COS2_THRESHOLD = 0.5

nonconstant_crm = [c for c in crm_cols if c not in constant_train_crm]
unique_rows = ~pd.Series(train_crm_id).duplicated().to_numpy()
crm_unique = train.loc[unique_rows, nonconstant_crm].reset_index(drop=True)
print(f"Matrix for the analysis: {crm_unique.shape[0]:,} unique subscriptions x {crm_unique.shape[1]} CRM columns")

Matrix for the analysis: 67,434 unique subscriptions x 743 CRM columns


#### 6.1 TruncatedSVD

In [12]:
start = time.time()
X_unique = csr_matrix(crm_unique.to_numpy(dtype=np.float64))
svd = TruncatedSVD(n_components=N_COMPONENTS, algorithm="arpack", random_state=RANDOM_STATE)  # exact solver
components_scores = svd.fit_transform(X_unique)

# cos2 of a variable = squared cosine between its column x_j and the space of the kept components:
# ||projection of x_j on the components||^2 / ||x_j||^2  (0 = not represented, 1 = fully represented)
orthonormal_basis, _ = np.linalg.qr(components_scores)
projected_norms = (np.asarray(X_unique.T @ orthonormal_basis) ** 2).sum(axis=1)
variable_norms = np.asarray(X_unique.multiply(X_unique).sum(axis=0)).ravel()
svd_cos2 = pd.Series(projected_norms / variable_norms, index=nonconstant_crm)

print(f"Explained variance with {N_COMPONENTS} components: {svd.explained_variance_ratio_.sum() * 100:.1f}%")
print(f"Time: {time.time() - start:.0f}s")
svd_cos2.describe()

Explained variance with 100 components: 77.0%
Time: 3s


count    743.000000
mean       0.209495
std        0.353914
min        0.002110
25%        0.016729
50%        0.019030
75%        0.112270
max        0.999969
dtype: float64

#### 6.2 Multiple Correspondence Analysis (MCA)

MCA turns each 0/1 variable into two categories (`__0` and `__1`). For a variable with two categories, both categories have the same cos², so the value of each variable is taken from its categories.

In [13]:
start = time.time()
mca = prince.MCA(n_components=N_COMPONENTS, correction="benzecri", random_state=RANDOM_STATE)
mca = mca.fit(crm_unique)

category_cos2 = mca.column_cosine_similarities(crm_unique).sum(axis=1)
# Category names look like "CRM_1533_PACK__1": remove the "__0" / "__1" ending to get the variable
category_variable = [re.sub(r"_{1,2}[01]$", "", str(name)) for name in category_cos2.index]
mca_cos2 = category_cos2.groupby(category_variable).max().reindex(nonconstant_crm)

print(f"Time: {time.time() - start:.0f}s")
mca_cos2.describe()

Time: 32s


count    743.000000
mean       0.252963
std        0.230440
min        0.033885
25%        0.115738
50%        0.137600
75%        0.289442
max        0.965165
dtype: float64

#### 6.3 How many CRM variables are well represented?

In [14]:
representation = pd.DataFrame({
    "Technique": ["TruncatedSVD", "MCA"],
    "Components": [N_COMPONENTS, N_COMPONENTS],
    "CRM variables": [len(svd_cos2), len(mca_cos2)],
    "Well represented (cos² ≥ 0.5)": [int((svd_cos2 >= COS2_THRESHOLD).sum()), int((mca_cos2 >= COS2_THRESHOLD).sum())],
})
representation["Share well represented"] = (
    representation["Well represented (cos² ≥ 0.5)"] / representation["CRM variables"]
).map("{:.1%}".format)
representation

,Technique,Components,CRM variables,Well represented (cos² ≥ 0.5),Share well represented
0,TruncatedSVD,100,743,150,20.2%
1,MCA,100,743,125,16.8%


**Decision:** the dimensionality reduction is kept as an analysis only. If a large share of the CRM variables is not well represented, replacing the original columns with the components would lose information about those options, and one wrong option is enough to get the whole bill wrong (EMR). The models in section 8 therefore use the original CRM columns. The table above shows the share for our data.

### 7. Validation design

We compare models on a **practice exam** (validation set) and use `solution.csv` only once, at the end.

· **Grouped split (80% / 20%):** all customers with the same subscription go to the same side. Otherwise the model could see the answer of a validation customer during training, and the score would look better than it really is.
· **Cleaning inside the training part only:** the cleaning uses the bills, so it is applied only to the 80%. The 20% keeps the original rows, just like the test set.
· **Same judge for everyone:** every model and every cleaning version is scored on the same validation rows with the same function.

In [15]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
fit_idx, validation_idx = next(splitter.split(train, groups=train_crm_id))
fit_part = train.iloc[fit_idx].reset_index(drop=True)
validation = train.iloc[validation_idx].reset_index(drop=True)
y_validation = validation[bil_cols].to_numpy(dtype=np.uint8)

assert not set(train_crm_id[fit_idx]) & set(train_crm_id[validation_idx])
print(f"Training part: {len(fit_part):,} rows | Validation: {len(validation):,} rows")


def score_predictions(predictions, truth):
    errors = np.asarray(predictions) != truth
    return {
        "EMR": np.all(~errors, axis=1).mean(),
        "Hamming loss": errors.mean(),
        "Mean wrong items per customer": errors.sum(axis=1).mean(),
    }

Training part: 153,380 rows | Validation: 34,062 rows


### 8. Models

The same six models are trained on each cleaning version:

| Model | Idea |
|---|---|
| Baseline | Gives everyone the most common bill. The score to beat. |
| k-Nearest Neighbor (k = 1) | Copies the bill of the most similar training subscription. |
| Binary Relevance | One decision tree per BIL item (731 trees). |
| Multi-output decision tree | One tree that predicts the 731 items together. |
| Association rules | For each BIL item, uses the single CRM option with the strongest phi association. |
| Association + targeted logistic | Association rules, plus a small logistic regression for the 15 hardest BIL items. |

The data is given to the tree and neighbor models as a **sparse matrix** (it stores only the 1s), which gives the same result much faster.

In [16]:
def association_predict(rows, weights, features, predict_rows):
    # For each BIL item: copy (or invert) the CRM option with the strongest phi association
    x_train = rows[features].to_numpy(dtype=np.uint8)
    y_train = rows[bil_cols].to_numpy(dtype=np.uint8)
    phi = phi_matrix(x_train, y_train, weights)
    best_feature = np.abs(phi).argmax(axis=0)
    best_phi = phi[best_feature, np.arange(len(bil_cols))]

    predictions = predict_rows[features].to_numpy(dtype=np.uint8)[:, best_feature].copy()
    negative = best_phi < 0
    predictions[:, negative] = 1 - predictions[:, negative]

    # Items with no association: use the weighted majority value of the item
    no_association = np.abs(best_phi) <= 1e-12
    target_majority = ((weights @ y_train) >= weights.sum() / 2).astype(np.uint8)
    predictions[:, no_association] = target_majority[no_association]
    return predictions


def find_hard_targets(raw_rows, cleaning, n_targets=15):
    # Inner grouped split inside the training rows: find the BIL items where
    # the association rules make the most errors
    groups = row_ids(raw_rows, crm_cols)
    inner = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE + 1)
    inner_fit, inner_check = next(inner.split(raw_rows, groups=groups))
    inner_rows, inner_weights, inner_features = cleaning(raw_rows.iloc[inner_fit])
    check_rows = raw_rows.iloc[inner_check]
    inner_predictions = association_predict(inner_rows, inner_weights, inner_features, check_rows)
    error_rate = (inner_predictions != check_rows[bil_cols].to_numpy(dtype=np.uint8)).mean(axis=0)
    return [bil_cols[i] for i in np.argsort(error_rate)[-n_targets:][::-1]]


def targeted_logistic_predict(rows, weights, features, predict_rows, hard_targets, n_features=8):
    predictions = association_predict(rows, weights, features, predict_rows)
    x_train = rows[features].to_numpy(dtype=np.uint8)
    phi = phi_matrix(x_train, rows[hard_targets].to_numpy(dtype=np.uint8), weights)
    for position, target in enumerate(hard_targets):
        y_target = rows[target].to_numpy(dtype=np.uint8)
        if np.unique(y_target).size < 2:
            continue
        # The 8 CRM options most associated with this BIL item
        selected = [features[i] for i in np.argsort(np.abs(phi[:, position]))[-n_features:]]
        model = LogisticRegression(C=0.5, solver="liblinear", max_iter=300, random_state=RANDOM_STATE)
        model.fit(rows[selected].to_numpy(dtype=np.uint8), y_target, sample_weight=weights)
        predictions[:, bil_cols.index(target)] = model.predict(predict_rows[selected].to_numpy(dtype=np.uint8))
    return predictions


def fit_predict(model_name, rows, weights, features, predict_rows, hard_targets=None):
    # Train one model on the cleaned rows and predict the 731 BIL items of predict_rows
    x_train = csr_matrix(rows[features].to_numpy(dtype=np.uint8))
    y_train = rows[bil_cols].to_numpy(dtype=np.uint8)
    x_predict = csr_matrix(predict_rows[features].to_numpy(dtype=np.uint8))

    if model_name == "Baseline":
        bill_ids = row_ids(rows, bil_cols)
        most_common_row = pd.Series(bill_ids).value_counts().index[0]
        bill = y_train[np.flatnonzero(bill_ids == most_common_row)[0]]
        predictions = np.tile(bill, (len(predict_rows), 1))
    elif model_name == "k-Nearest Neighbor (k = 1)":
        neighbors = NearestNeighbors(n_neighbors=1).fit(x_train)
        nearest = neighbors.kneighbors(x_predict, return_distance=False)[:, 0]
        predictions = y_train[nearest]
    elif model_name == "Binary Relevance":
        model = MultiOutputClassifier(DecisionTreeClassifier(random_state=RANDOM_STATE), n_jobs=-1)
        predictions = model.fit(x_train, y_train).predict(x_predict)
    elif model_name == "Multi-output decision tree":
        model = DecisionTreeClassifier(max_depth=16, min_samples_leaf=50, max_features="sqrt",
                                       random_state=RANDOM_STATE)
        predictions = model.fit(x_train, y_train).predict(x_predict)
    elif model_name == "Association rules":
        predictions = association_predict(rows, weights, features, predict_rows)
    elif model_name == "Association + targeted logistic":
        predictions = targeted_logistic_predict(rows, weights, features, predict_rows, hard_targets)
    else:
        raise ValueError(model_name)
    return np.asarray(predictions, dtype=np.uint8)


MODEL_NAMES = [
    "Baseline",
    "k-Nearest Neighbor (k = 1)",
    "Binary Relevance",
    "Multi-output decision tree",
    "Association rules",
    "Association + targeted logistic",
]

#### 8.1 Train every model on each cleaning version and score it on the validation set

In [17]:
results = []
for cleaning_name, cleaning in CLEANINGS.items():
    rows, weights, features = cleaning(fit_part)
    hard_targets = find_hard_targets(fit_part, cleaning)
    for model_name in MODEL_NAMES:
        start = time.time()
        predictions = fit_predict(model_name, rows, weights, features, validation, hard_targets)
        assert predictions.shape == y_validation.shape
        result = {"Cleaning": cleaning_name, "Model": model_name, **score_predictions(predictions, y_validation)}
        result["Seconds"] = round(time.time() - start, 1)
        results.append(result)
        print(f"{cleaning_name:14} | {model_name:32} | EMR = {result['EMR']:.2%} | {result['Seconds']}s")

results_table = pd.DataFrame(results)

A – strict     | Baseline                         | EMR = 0.01% | 0.7s
A – strict     | k-Nearest Neighbor (k = 1)       | EMR = 0.41% | 28.8s
A – strict     | Binary Relevance                 | EMR = 86.58% | 264.6s
A – strict     | Multi-output decision tree       | EMR = 4.59% | 7.2s
A – strict     | Association rules                | EMR = 86.49% | 2.1s
A – strict     | Association + targeted logistic  | EMR = 86.70% | 3.8s
B – majority   | Baseline                         | EMR = 0.01% | 1.0s
B – majority   | k-Nearest Neighbor (k = 1)       | EMR = 0.50% | 35.1s
B – majority   | Binary Relevance                 | EMR = 86.21% | 356.7s
B – majority   | Multi-output decision tree       | EMR = 1.99% | 5.9s
B – majority   | Association rules                | EMR = 86.13% | 2.3s
B – majority   | Association + targeted logistic  | EMR = 86.52% | 4.0s


### 9. Model and cleaning selection

The best combination is the one with the **highest validation EMR** (if two are equal, the lower Hamming loss wins). The choice is made **only** with the validation set; `solution.csv` is not used here.

In [18]:
emr_table = (results_table.pivot(index="Model", columns="Cleaning", values="EMR").reindex(MODEL_NAMES) * 100).round(2)
print("Validation EMR (%) by model and cleaning:")
display(emr_table)

ranking = results_table.sort_values(["EMR", "Hamming loss"], ascending=[False, True]).reset_index(drop=True)
best_cleaning = ranking.loc[0, "Cleaning"]
best_model = ranking.loc[0, "Model"]
print(f"Selected: cleaning {best_cleaning} + {best_model} (validation EMR = {ranking.loc[0, 'EMR']:.2%})")
ranking

Validation EMR (%) by model and cleaning:


Cleaning,A – strict,B – majority
Model,,
Baseline,0.01,0.01
k-Nearest Neighbor (k = 1),0.41,0.50
Binary Relevance,86.58,86.21
Multi-output decision tree,4.59,1.99
Association rules,86.49,86.13
Association + targeted logistic,86.70,86.52


Selected: cleaning A – strict + Association + targeted logistic (validation EMR = 86.70%)


,Cleaning,Model,EMR,Hamming loss,Mean wrong items per customer,Seconds
0,A – strict,Association + targeted logistic,0.867007,0.002456,1.795226,3.8
1,A – strict,Binary Relevance,0.865833,0.002895,2.116259,264.6
2,B – majority,Association + targeted logistic,0.865216,0.002469,1.804826,4.0
3,A – strict,Association rules,0.864893,0.002473,1.807498,2.1
4,B – majority,Binary Relevance,0.862134,0.003121,2.281134,356.7
5,B – majority,Association rules,0.861341,0.002491,1.821208,2.3
6,A – strict,Multi-output decision tree,0.045916,0.006966,5.091979,7.2
7,B – majority,Multi-output decision tree,0.019934,0.007082,5.176678,5.9
8,B – majority,k-Nearest Neighbor (k = 1),0.004962,0.003035,2.218807,35.1
9,A – strict,k-Nearest Neighbor (k = 1),0.004081,0.003185,2.328431,28.8


### 10. Final model: train on all training data and predict the test set

The selected cleaning is applied to the **full** training set, the selected model is trained on it, and the test customers are predicted. The submission follows the answer-file format: `MSISDN` and one 731-character `Bill_Conf` text per customer.

In [19]:
start = time.time()
final_cleaning = CLEANINGS[best_cleaning]
final_rows, final_weights, final_features = final_cleaning(train)
final_hard_targets = find_hard_targets(train, final_cleaning) if best_model == "Association + targeted logistic" else None
final_predictions = fit_predict(best_model, final_rows, final_weights, final_features, test, final_hard_targets)

assert final_predictions.shape == (len(test), len(bil_cols))
submission = pd.DataFrame({
    id_col: test[id_col].to_numpy(),
    "Bill_Conf": ["".join(row.astype(str)) for row in final_predictions],
})
assert submission["Bill_Conf"].str.fullmatch(r"[01]{731}").all()
submission.to_csv("submission.csv", index=False)

print(f"Trained on {len(final_rows):,} rows with {len(final_features)} CRM features")
print(f"Saved submission.csv: {submission.shape} | Time: {time.time() - start:.0f}s")
submission.head()

Trained on 62,622 rows with 743 CRM features
Saved submission.csv: (97100, 2) | Time: 46s


,MSISDN,Bill_Conf
0,db67b03cf9fb8ea96077cd34c7a28139,1000000000000000000000000000000000000000000000...
1,2ac0d310d5cf1a96587af614d37b8b54,1000000000100000000000000000000100000000000000...
2,a8f289fbd263794ac74059a18c1d10a5,1100000110111100000000000000000000010000000000...
3,5a53a1c85ccc7e794c90c75b75b8d3cb,1110000110111010100000000000000000010000000000...
4,484b400226b8894377e12204219283fc,1000000000000000000000000000000000000000000000...


### 11. Final check against `solution.csv` (only once)

This is the final exam. It is only a diagnostic: nothing in the notebook is tuned with these answers.

In [20]:
solution = pd.read_csv("solution.csv", dtype=str)
aligned = submission.set_index(id_col).reindex(solution[id_col])
assert aligned["Bill_Conf"].notna().all()

predicted_bits = np.array([list(bill) for bill in aligned["Bill_Conf"]], dtype=np.uint8)
true_bits = np.array([list(bill) for bill in solution["Bill_Conf"]], dtype=np.uint8)
test_report = pd.DataFrame([{
    "Cleaning": best_cleaning,
    "Model": best_model,
    "Test customers": len(solution),
    **score_predictions(predicted_bits, true_bits),
}])
test_report

,Cleaning,Model,Test customers,EMR,Hamming loss,Mean wrong items per customer
0,A – strict,Association + targeted logistic,97100,0.962987,0.000081,0.059423
